# 03 Modeling

**Objective:** Compare baseline and machine learning models for two tasks: regression of `CarbonEmission` (main metric MAE) and classification of `high_emission` (main metric PR-AUC). Preprocessing is part of every pipeline, so it is fitted only on training folds during cross-validation.

## 0. Setup
Imports.

In [1]:
import pandas as pd
import numpy as np
import ast

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.base import clone
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 3.1.1
NumPy version: 2.5.4


## 1. Data loading
Load the train and test sets created in `02_split`. The test set is not used until the final evaluation.

In [2]:
RANDOM_STATE = 42
TARGET_REG = "CarbonEmission"
TARGET_CLF = "high_emission"

train = pd.read_csv("../data/processed/train.csv")
test = pd.read_csv("../data/processed/test.csv")

print("\nTrain data shape with target:", train.shape)
print("Test data shape with target:", test.shape)


Train data shape with target: (8000, 21)
Test data shape with target: (2000, 21)


## 2. Features and targets
Separate the features from the two targets and check the shapes.

In [3]:
X_train = train.drop(columns=[TARGET_REG, TARGET_CLF])
y_reg_train = train[TARGET_REG]
y_clf_train = train[TARGET_CLF]

X_test = test.drop(columns=[TARGET_REG, TARGET_CLF])
y_reg_test = test[TARGET_REG]
y_clf_test = test[TARGET_CLF]

assert list(X_train.columns) == list(X_test.columns)
assert TARGET_REG not in X_train.columns and TARGET_CLF not in X_train.columns
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Class 1 share, train:", y_clf_train.mean().round(3), "| test:", y_clf_test.mean().round(3))

Train: (8000, 19) Test: (2000, 19)
Class 1 share, train: 0.2 | test: 0.2


## 3. Column groups
Numerical, text and list columns, detected from the training features.

In [4]:
obj_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

list_cols = [
    col for col in obj_cols
    if X_train[col].dropna().astype(str).str.strip().str.startswith("[").any()
]
str_cols = [col for col in obj_cols if col not in list_cols]
num_cols = X_train.select_dtypes(include="number").columns.tolist()

print("Number of numerical columns:", len(num_cols))
print("Number of string columns:", len(str_cols))
print("Number of list columns:", len(list_cols))
print("Sum of all columns:", len(num_cols) + len(str_cols) + len(list_cols))
print("List columns:", list_cols)

Number of numerical columns: 6
Number of string columns: 11
Number of list columns: 2
Sum of all columns: 19
List columns: ['Recycling', 'Cooking_With']


## 4. Custom transformer
`ListToMLB` turns a column of lists into multi-hot columns.

In [5]:
class ListToMLB(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb_ = MultiLabelBinarizer()

    @staticmethod
    def _parse_cell(x):
        if isinstance(x, list):
            return x
        if isinstance(x, float) and pd.isna(x):
            return []
        if isinstance(x, str):
            s = x.strip()
            if s.startswith("[") and s.endswith("]"):
                try:
                    return ast.literal_eval(s)
                except Exception:
                    return []
            return []
        return []

    def fit(self, X, y=None):
        col = pd.Series(np.asarray(X).ravel()).apply(self._parse_cell)
        self.mlb_.fit(col)
        return self

    def transform(self, X):
        col = pd.Series(np.asarray(X).ravel()).apply(self._parse_cell)
        return self.mlb_.transform(col)

    def get_feature_names_out(self, input_features=None):
        return np.array([f"list__{c}" for c in self.mlb_.classes_], dtype=object)

## 5. Preprocessing
Numerical: median imputation and `RobustScaler`. Text: constant imputation and one-hot encoding. List columns: one `ListToMLB` per column.

In [6]:
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])

str_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(sparse_output=False, handle_unknown="ignore", dtype="bool"))
])

list_transforms = [
    (f"list_{i}", Pipeline([("list", ListToMLB())]), [col])
    for i, col in enumerate(list_cols)
]

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("str", str_pipe, str_cols),
        *list_transforms
    ],
    remainder="drop"
)

preprocess


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('str', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_n

## 6. Helper functions
Functions that run cross-validation: one to choose k for k-NN, and one each for classification (PR-AUC, ROC-AUC, F1, recall, accuracy) and regression (MAE, RMSE).

In [7]:
def find_best_k(preprocess, X, y, cv, k_values, task):
    results = {}

    for k in k_values:
        if task == "clf":
            model = KNeighborsClassifier(n_neighbors=k)
            scoring = {
                "pr_auc": "average_precision",
                "f1": "f1"
            }

        elif task == "reg":
            model = KNeighborsRegressor(n_neighbors=k)
            scoring = {
                "mae": "neg_mean_absolute_error"
            }

        else:
            raise ValueError("Task must be 'clf' or 'reg'")

        pipe = Pipeline([
            ("pre", clone(preprocess)),
            ("model", model)
        ])

        scores = cross_validate(
            pipe, X, y,
            cv=cv,
            scoring=scoring,
            n_jobs=-1
        )

        results[k] = {}

        for metric in scoring:
            value = scores[f"test_{metric}"].mean()

            if metric == "mae":
                value = -value

            results[k][metric] = value

    return results

In [8]:
def evaluate_clf(model, X, y, cv):
    scoring = {
        "pr_auc": "average_precision",
        "accuracy": "accuracy",
        "f1": "f1",
        "recall": "recall",
        "roc_auc": "roc_auc"
    }

    results = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    return {
        metric: results[f"test_{metric}"]
        for metric in scoring
    }

In [9]:
def evaluate_reg(model, X, y, cv):
    scoring = {
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error"
    }

    results = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    return {
        metric: -results[f"test_{metric}"]
        for metric in scoring
    }

## 7. Cross-validation setup
5 folds, stratified for classification. Every model uses the same folds, so results are comparable.

In [10]:
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 8. Baseline models
Dummy models (most frequent class, median) give the reference scores every model should beat.

In [11]:
pipe_dummy_clf = Pipeline([("pre", clone(preprocess)), ("clf", DummyClassifier(strategy="most_frequent"))])
pipe_dummy_reg = Pipeline([("pre", clone(preprocess)), ("reg", DummyRegressor(strategy="median"))])

clf_dummy_results = evaluate_clf(pipe_dummy_clf, X_train, y_clf_train, cv_clf)
reg_dummy_results = evaluate_reg(pipe_dummy_reg, X_train, y_reg_train, cv_reg)

print("Dummy: Accuracy:", clf_dummy_results["accuracy"].mean().round(3), "| ROC-AUC:", clf_dummy_results["roc_auc"].mean().round(3), "| PR-AUC:", clf_dummy_results["pr_auc"].mean().round(3), "| F1:", clf_dummy_results["f1"].mean().round(3), "| Recall:", clf_dummy_results["recall"].mean().round(3))
print("Dummy: MAE:", reg_dummy_results["mae"].mean().round(3), "| RMSE:", reg_dummy_results["rmse"].mean().round(3))

Dummy: Accuracy: 0.8 | ROC-AUC: 0.5 | PR-AUC: 0.2 | F1: 0.0 | Recall: 0.0
Dummy: MAE: 765.988 | RMSE: 1035.017


## 9. Linear models
Logistic regression and linear regression as simple models.

In [12]:
pipe_log = Pipeline([("pre", clone(preprocess)), ("clf", LogisticRegression(max_iter=1000))])
pipe_lin = Pipeline([("pre", clone(preprocess)), ("reg", LinearRegression())])

log_clf_result = evaluate_clf(pipe_log, X_train, y_clf_train, cv_clf)
lin_reg_result = evaluate_reg(pipe_lin, X_train, y_reg_train, cv_reg)

print("Logistic Regression: Accuracy:", log_clf_result["accuracy"].mean().round(3), "| ROC-AUC:", log_clf_result["roc_auc"].mean().round(3), "| PR-AUC:", log_clf_result["pr_auc"].mean().round(3), "| F1:", log_clf_result["f1"].mean().round(3), "| Recall:", log_clf_result["recall"].mean().round(3))
print("Linear Regression: MAE:", lin_reg_result["mae"].mean().round(3), "| RMSE:", lin_reg_result["rmse"].mean().round(3))

Logistic Regression: Accuracy: 0.964 | ROC-AUC: 0.992 | PR-AUC: 0.974 | F1: 0.906 | Recall: 0.88
Linear Regression: MAE: 176.482 | RMSE: 263.431


## 10. k-NN

### 10.1 Choosing k
Odd values of k up to 21 plus a few larger ones, chosen by PR-AUC (classification) and MAE (regression).

In [13]:
k_values = list(range(1, 22, 2)) + [31, 41, 51, 61]

clf_k_results = find_best_k(preprocess, X_train, y_clf_train, cv_clf, k_values, "clf")

reg_k_results = find_best_k(preprocess, X_train, y_reg_train, cv_reg, k_values, "reg")

best_k_clf_pr = max(clf_k_results, key=lambda k: clf_k_results[k]["pr_auc"])
best_k_clf_f1 = max(clf_k_results, key=lambda k: clf_k_results[k]["f1"])

best_k_reg = min(reg_k_results, key=lambda k: reg_k_results[k]["mae"])

best_k_clf = best_k_clf_pr

print("PR-AUC:", {k: round(float(v["pr_auc"]), 3) for k, v in clf_k_results.items()})
print("F1:", {k: round(float(v["f1"]), 3) for k, v in clf_k_results.items()})

print("Best k (PR-AUC):", best_k_clf_pr)
print("Best k (F1):", best_k_clf_f1)

print()

print("MAE:", {k: round(float(v["mae"]), 3) for k, v in reg_k_results.items()})
print("Best k (regression):", best_k_reg)

PR-AUC: {1: 0.478, 3: 0.678, 5: 0.752, 7: 0.788, 9: 0.807, 11: 0.815, 13: 0.823, 15: 0.827, 17: 0.831, 19: 0.834, 21: 0.836, 31: 0.841, 41: 0.846, 51: 0.846, 61: 0.843}
F1: {1: 0.635, 3: 0.69, 5: 0.714, 7: 0.719, 9: 0.726, 11: 0.716, 13: 0.711, 15: 0.71, 17: 0.71, 19: 0.701, 21: 0.704, 31: 0.691, 41: 0.696, 51: 0.686, 61: 0.686}
Best k (PR-AUC): 51
Best k (F1): 9

MAE: {1: 556.248, 3: 424.55, 5: 398.231, 7: 387.658, 9: 383.077, 11: 382.049, 13: 381.415, 15: 381.266, 17: 381.831, 19: 384.0, 21: 385.947, 31: 393.218, 41: 398.073, 51: 403.892, 61: 409.966}
Best k (regression): 15


### 10.2 k-NN results
Cross-validated scores with the chosen k.

In [14]:
pipe_knn_reg = Pipeline([("prep", clone(preprocess)), ("reg", KNeighborsRegressor(n_neighbors=best_k_reg))])
pipe_knn_clf = Pipeline([("prep", clone(preprocess)), ("clf", KNeighborsClassifier(n_neighbors=best_k_clf))])

clf_knn_results = evaluate_clf(pipe_knn_clf, X_train, y_clf_train, cv_clf)
reg_knn_results = evaluate_reg(pipe_knn_reg, X_train, y_reg_train, cv_reg)

print("k-NN classification: Accuracy:", clf_knn_results["accuracy"].mean().round(3), "| ROC-AUC:", clf_knn_results["roc_auc"].mean().round(3), "| PR-AUC:", clf_knn_results["pr_auc"].mean().round(3), "| F1:", clf_knn_results["f1"].mean().round(3), "| Recall:", clf_knn_results["recall"].mean().round(3))
print(f"k-NN regresja (k={best_k_reg}) MAE:", reg_knn_results["mae"].mean().round(3), "| RMSE:", reg_knn_results["rmse"].mean().round(3))

k-NN classification: Accuracy: 0.894 | ROC-AUC: 0.954 | PR-AUC: 0.846 | F1: 0.686 | Recall: 0.576
k-NN regresja (k=15) MAE: 381.266 | RMSE: 509.778


## 11. Decision tree
A single tree with default settings.

In [15]:
pipe_tree_clf = Pipeline([("pre", clone(preprocess)), ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))])
pipe_tree_reg = Pipeline([("pre", clone(preprocess)), ("reg", DecisionTreeRegressor(random_state=RANDOM_STATE))])

clf_tree_results = evaluate_clf(pipe_tree_clf, X_train, y_clf_train, cv_clf)
reg_tree_results = evaluate_reg(pipe_tree_reg, X_train, y_reg_train, cv_reg)

print("Decision Tree classification: Accuracy:", clf_tree_results["accuracy"].mean().round(3), "| ROC-AUC:", clf_tree_results["roc_auc"].mean().round(3), "| PR-AUC:", clf_tree_results["pr_auc"].mean().round(3), "| F1:", clf_tree_results["f1"].mean().round(3), "| Recall:", clf_tree_results["recall"].mean().round(3))
print("Decision Tree regression: MAE:", reg_tree_results["mae"].mean().round(3), "| RMSE:", reg_tree_results["rmse"].mean().round(3))

Decision Tree classification: Accuracy: 0.904 | ROC-AUC: 0.855 | PR-AUC: 0.629 | F1: 0.763 | Recall: 0.772
Decision Tree regression: MAE: 343.699 | RMSE: 457.932


## 12. Random forest
Forest of 200 trees.

In [16]:
pipe_rf_clf = Pipeline([("prep", clone(preprocess)), ("clf", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))])
pipe_rf_reg = Pipeline([("prep", clone(preprocess)), ("reg", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))])

clf_rf_results = evaluate_clf(pipe_rf_clf, X_train, y_clf_train, cv_clf)
reg_rf_results = evaluate_reg(pipe_rf_reg, X_train, y_reg_train, cv_reg)

print("Random Forest classification: Accuracy:", clf_rf_results["accuracy"].mean().round(3), "| ROC-AUC:", clf_rf_results["roc_auc"].mean().round(3), "| PR-AUC:", clf_rf_results["pr_auc"].mean().round(3), "| F1:", clf_rf_results["f1"].mean().round(3), "| Recall:", clf_rf_results["recall"].mean().round(3))
print("Random Forest regression: MAE:", reg_rf_results["mae"].mean().round(3), "| RMSE:", reg_rf_results["rmse"].mean().round(3))

Random Forest classification: Accuracy: 0.94 | ROC-AUC: 0.979 | PR-AUC: 0.935 | F1: 0.837 | Recall: 0.764
Random Forest regression: MAE: 220.3 | RMSE: 293.425


## 13. Gradient boosting
Boosted ensemble of 200 trees.

In [17]:
pipe_gb_clf = Pipeline([("prep", clone(preprocess)), ("clf", GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE))])
pipe_gb_reg = Pipeline([("prep", clone(preprocess)), ("reg", GradientBoostingRegressor(n_estimators=200, random_state=RANDOM_STATE))])

clf_gb_results = evaluate_clf(pipe_gb_clf, X_train, y_clf_train, cv_clf)
reg_gb_results = evaluate_reg(pipe_gb_reg, X_train, y_reg_train, cv_reg)

print("Gradient Boosting classification: Accuracy:", clf_gb_results["accuracy"].mean().round(3), "| ROC-AUC:", clf_gb_results["roc_auc"].mean().round(3), "| PR-AUC:", clf_gb_results["pr_auc"].mean().round(3), "| F1:", clf_gb_results["f1"].mean().round(3), "| Recall:", clf_gb_results["recall"].mean().round(3))
print("Gradient Boosting regression: MAE:", reg_gb_results["mae"].mean().round(3), "| RMSE:", reg_gb_results["rmse"].mean().round(3))

Gradient Boosting classification: Accuracy: 0.962 | ROC-AUC: 0.993 | PR-AUC: 0.973 | F1: 0.902 | Recall: 0.867
Gradient Boosting regression: MAE: 106.425 | RMSE: 146.975


## 14. Model comparison
Average cross-validated scores of all models, sorted from best to worst: PR-AUC and F1 for classification, MAE and RMSE for regression.

In [18]:
clf_results = {
    "Dummy": clf_dummy_results,
    "Logistic regression": log_clf_result,
    "k-NN": clf_knn_results,
    "Decision tree": clf_tree_results,
    "Random forest": clf_rf_results,
    "Gradient boosting": clf_gb_results,
}

clf_table = pd.DataFrame(
    {name: {"PR-AUC": r["pr_auc"].mean(), "F1": r["f1"].mean()} for name, r in clf_results.items()}
).T.round(3).sort_values("PR-AUC", ascending=False)

clf_table

,PR-AUC,F1
Logistic regression,0.974,0.906
Gradient boosting,0.973,0.902
Random forest,0.935,0.837
k-NN,0.846,0.686
Decision tree,0.629,0.763
Dummy,0.200,0.000


In [19]:
reg_results = {
    "Dummy": reg_dummy_results,
    "Linear regression": lin_reg_result,
    "k-NN": reg_knn_results,
    "Decision tree": reg_tree_results,
    "Random forest": reg_rf_results,
    "Gradient boosting": reg_gb_results,
}

reg_table = pd.DataFrame(
    {name: {"MAE": r["mae"].mean(), "RMSE": r["rmse"].mean()} for name, r in reg_results.items()}
).T.round(1).sort_values("MAE")

reg_table

,MAE,RMSE
Gradient boosting,106.4,147.0
Linear regression,176.5,263.4
Random forest,220.3,293.4
Decision tree,343.7,457.9
k-NN,381.3,509.8
Dummy,766.0,1035.0


## 15. Conclusions
- All models are compared with the same cross-validation folds on the training set. The test set has not been used yet.
- The dummy classifier reaches about 0.8 accuracy but only 0.2 PR-AUC (the class 1 share), so accuracy is not a useful metric here.
- Classification: logistic regression is the best model (PR-AUC 0.974, F1 0.906), but gradient boosting is almost identical (0.973 and 0.902). Both are far above the baseline (PR-AUC 0.200, F1 0.000).
- Regression: gradient boosting is the best model (MAE 106.4, RMSE 147.0). Its MAE is about 86% lower than the dummy baseline (766.0) and clearly lower than linear regression (176.5).
- k-NN (k = [X]) is among the weakest models in both tasks (PR-AUC 0.846, F1 0.686, MAE 381.3), likely because the many one-hot columns make distances less informative.
- Models use default or near-default settings. Only k in k-NN was chosen by cross-validation, and no other tuning has been done yet.
- The dataset is synthetic and its target is calculated from the features, so the very high scores (especially for linear models) likely reflect how the target was generated, not performance on real-world data.

### Next steps
- Tune the best models (logistic regression and gradient boosting for classification, gradient boosting for regression) to check whether any of them improves.
- Evaluate the final models once on the test set, including a confusion matrix for classification.